# Homework 1: Retrieval-Based Chatbot (Russian)


In [22]:
!pip install -q transformers datasets evaluate accelerate sentencepiece
!pip install -q sentence-transformers rank-bm25

!pip install -q faiss-gpu 2>/dev/null || pip install -q faiss-cpu


In [38]:
import time
import re
import urllib.request
import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tabulate import tabulate

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Local GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU")


Local GPU: NVIDIA GeForce RTX 3090


## 1. Data
Download `Master and Margarita` text and build a retrieval corpus.


In [24]:
with open("/content/LLM_1/Bulgakov_Mihail_Master_i_Margarita.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

# Basic cleanup
raw_text = raw_text.replace("\r", "\n")
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text)

In [25]:
# Split into paragraphs and keep meaningful ones
paragraphs = [p.strip() for p in raw_text.split("\n\n")]
paragraphs = [p for p in paragraphs if len(p) >= 120 and not p.startswith("http")]

N_DOCS = min(10000, len(paragraphs))
documents = paragraphs[:N_DOCS]
titles = [f"Passage {i}" for i in range(N_DOCS)]

# Query set for automatic evaluation (self-retrieval setup)
N_QUERIES = min(200, N_DOCS)
queries = [re.split(r"[.!?]\s+", documents[i])[0][:120] for i in range(N_QUERIES)]
ground_truth = list(range(N_QUERIES))

print(f"Documents: {len(documents)}")
print(f"Queries: {len(queries)}")
if queries:
    print(f"Example query: {queries[0]}")


Documents: 2025
Queries: 200
Example query: Текст печатается в последней прижизненной редакции (рукописи хранятся в рукописном отделе Государственной библиотеки ССС


In [26]:
paragraphs[0]

'Текст печатается в последней прижизненной редакции (рукописи хранятся в рукописном отделе Государственной библиотеки СССР имени В. И. Ленина), а также с исправлениями и дополнениями, сделанными под диктовку писателя его женой, Е. С. Булгаковой.'

## 2. Keyword Baseline (BM25)

In [27]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)


def bm25_search(query: str, top_k: int = 10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices.tolist(), scores[top_indices].tolist()


def bm25_answer_user(query: str, top_k: int = 3):
    idxs, scores = bm25_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 3. Dense Retrieval (Sentence-Transformers)

In [28]:
retriever = SentenceTransformer("intfloat/multilingual-e5-small", device=str(device))

docs_with_prefix = [f"passage: {doc}" for doc in documents]

t0 = time.time()
doc_embeddings = retriever.encode(
    docs_with_prefix,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {doc_embeddings.shape}")
print(f"Encoding time: {time.time() - t0:.2f} sec")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings shape: (2025, 384)
Encoding time: 1.35 sec


In [29]:
def dense_search(query: str, top_k: int = 10):
    query_emb = retriever.encode(f"query: {query}", normalize_embeddings=True)
    scores = doc_embeddings @ query_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices.tolist(), scores[top_indices].tolist()


def dense_answer_user(query: str, top_k: int = 3):
    idxs, scores = dense_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 4. FAISS Index

In [30]:
d = doc_embeddings.shape[1]
index_flat = faiss.IndexFlatIP(d)
index_flat.add(doc_embeddings.astype(np.float32))

print(f"FAISS index vectors: {index_flat.ntotal}")

FAISS index vectors: 2025


In [31]:
def faiss_search(query: str, top_k: int = 10):
    query_emb = retriever.encode([f"query: {query}"], normalize_embeddings=True).astype(np.float32)
    scores, indices = index_flat.search(query_emb, top_k)
    return indices[0].tolist(), scores[0].tolist()


def faiss_answer_user(query: str, top_k: int = 3):
    idxs, scores = faiss_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 5. Metrics

In [32]:
def recall_at_k(results, ground_truth, k: int) -> float:
    hits = sum(1 for i, res in enumerate(results) if ground_truth[i] in res[:k])
    return hits / len(results)


def mrr(results, ground_truth) -> float:
    rr_sum = 0.0
    for i, res in enumerate(results):
        try:
            rank = res.index(ground_truth[i]) + 1
            rr_sum += 1.0 / rank
        except ValueError:
            pass
    return rr_sum / len(results)


def ndcg_at_k(results, ground_truth, k: int) -> float:
    ndcg_sum = 0.0
    for i, res in enumerate(results):
        dcg = 0.0
        for rank, doc_idx in enumerate(res[:k], 1):
            if doc_idx == ground_truth[i]:
                dcg += 1.0 / np.log2(rank + 1)
        ndcg_sum += dcg
    return ndcg_sum / len(results)

## 6. Evaluate BM25 vs Dense vs FAISS

In [33]:
bm25_results = []
dense_results = []
faiss_results = []

for q in queries:
    bm25_idxs, _ = bm25_search(q, top_k=20)
    dense_idxs, _ = dense_search(q, top_k=20)
    faiss_idxs, _ = faiss_search(q, top_k=20)
    bm25_results.append(bm25_idxs)
    dense_results.append(dense_idxs)
    faiss_results.append(faiss_idxs)

rows = []
all_methods = {
    "BM25": bm25_results,
    "Dense (brute-force)": dense_results,
    "Dense + FAISS Flat": faiss_results,
}

for name, res in all_methods.items():
    rows.append([
        name,
        f"{mrr(res, ground_truth):.3f}",
        f"{ndcg_at_k(res, ground_truth, 10):.3f}",
        f"{recall_at_k(res, ground_truth, 1):.3f}",
        f"{recall_at_k(res, ground_truth, 5):.3f}",
        f"{recall_at_k(res, ground_truth, 10):.3f}",
    ])

print(tabulate(rows, headers=["Method", "MRR", "NDCG@10", "R@1", "R@5", "R@10"], tablefmt="rounded_outline"))

╭─────────────────────┬───────┬───────────┬───────┬───────┬────────╮
│ Method              │   MRR │   NDCG@10 │   R@1 │   R@5 │   R@10 │
├─────────────────────┼───────┼───────────┼───────┼───────┼────────┤
│ BM25                │ 0.932 │     0.936 │  0.92 │ 0.945 │  0.95  │
│ Dense (brute-force) │ 0.886 │     0.899 │  0.86 │ 0.91  │  0.945 │
│ Dense + FAISS Flat  │ 0.886 │     0.899 │  0.86 │ 0.91  │  0.945 │
╰─────────────────────┴───────┴───────────┴───────┴───────┴────────╯


## 7. Inference Demo (with similarity scores)


In [34]:
user_query = "Понтий Пилат"

print("BM25:")
for row in bm25_answer_user(user_query, top_k=3):
    print(f"{row['rank']}. score={row['score']:.3f} | {row['title']}")

print("\nDense + FAISS:")
for row in faiss_answer_user(user_query, top_k=3):
    print(f"{row['rank']}. score={row['score']:.3f} | {row['title']}")


BM25:
1. score=11.494 | Passage 720
2. score=8.579 | Passage 744
3. score=7.285 | Passage 1701

Dense + FAISS:
1. score=0.865 | Passage 361
2. score=0.865 | Passage 475
3. score=0.858 | Passage 454


## 8. Qualitative Comparison: Vector Search vs Keyword Search
Use your own 3-5 queries and write short conclusions in markdown.


In [41]:
test_queries = [
    "что хорошо, что плохо",
    "опасности любви",
]

In [42]:
for q in test_queries:
    print("=" * 90)
    print(f"Query: {q}")

    bm25_top1 = bm25_answer_user(q, top_k=1)[0]
    faiss_top1 = faiss_answer_user(q, top_k=1)[0]

    print(f"\nBM25  (score={bm25_top1['score']:.3f}):\n{bm25_top1['text'][:300]}")
    print(f"\nDense (score={faiss_top1['score']:.3f}):\n{faiss_top1['text'][:300]}")

Query: что хорошо, что плохо

BM25  (score=8.744):
– Хорошо, хорошо, – фальшиво-ласково говорил Берлиоз и, подмигнув расстроенному поэту, которому вовсе не улыбалась мысль караулить сумасшедшего немца, устремился к тому выходу с Патриарших, что находится на углу Бронной и Ермолаевского переулка.

Dense (score=0.852):
– Ах! Оскорбление является обычной наградой за хорошую работу, – ответил Азазелло, – неужели вы слепы? Но прозрейте же скорей.
Query: опасности любви

BM25  (score=8.880):
И на всем его трудном пути невыразимо почему-то мучил вездесущий оркестр, под аккомпанемент которого тяжелый бас пел о своей любви к Татьяне.

Dense (score=0.846):
– Погоди... зайдем в этот дворик и условимся, а то я боюсь, что кто-нибудь из знакомых увидит меня и потом скажут, что я была с любовником на улице.


In [39]:
test_queries = [
    "человек смертен и внезапно смертен",
    "рукописи не горят",
    "трусость - самый страшный порок",
]

In [40]:
for q in test_queries:
    print("=" * 90)
    print(f"Query: {q}")

    bm25_top1 = bm25_answer_user(q, top_k=1)[0]
    faiss_top1 = faiss_answer_user(q, top_k=1)[0]

    print(f"\nBM25  (score={bm25_top1['score']:.3f}):\n{bm25_top1['text'][:300]}")
    print(f"\nDense (score={faiss_top1['score']:.3f}):\n{faiss_top1['text'][:300]}")

Query: человек смертен и внезапно смертен

BM25  (score=23.268):
А я черт знает чем занялся! Важное, в самом деле, происшествие – редактора журнала задавило! Да что от этого, журнал, что ли, закроется? Ну, что ж поделаешь: человек смертен и, как справедливо сказано было, внезапно смертен. Ну, царство небесное ему! Ну, будет другой редактор и даже, может быть, еще

Dense (score=0.862):
– Да, человек смертен, но это было бы еще полбеды. Плохо то, что он иногда внезапно смертен, вот в чем фокус! И вообще не может сказать, что он будет делать в сегодняшний вечер.
Query: рукописи не горят

BM25  (score=7.999):
– Простите, не поверю, – ответил Воланд, – этого быть не может. Рукописи не горят. – Он повернулся к Бегемоту и сказал: – Ну-ка, Бегемот, дай сюда роман.

Dense (score=0.844):
– Насчет ножа не беспокойся, нож вернут в лавку. А теперь мне нужно второе: покажи хартию, которую ты носишь с собой и где записаны слова Иешуа.
Query: трусость - самый страшный порок

BM25  (score=11.273):
Своб

## 8. Vector Search vs Keyword Search

**"человек смертен и внезапно смертен"**
BM25 нашёл отрывок, где персонаж пересказывает эту мысль
"как справедливо сказано было". Но даже мне читая эти ответы сложно было понять, что тут является наиболее правильным ответов. Dense нашёл
отрывок, где та же идея звучит от первого лица.
Какой из них "правильнее" зависит от того, что хочет пользователь,
но Dense нашёл более прямое высказывание.

**"рукописи не горят"**
BM25 сразу нашёл точную цитату Воланда. Мне кажется тут ответ Dense даже по смыслу не подходит.

**"трусость — самый страшный порок"**
Оба метода вернули один и тот же правильный отрывок.

** "что хорошо, что плохо" и "опасности любви" **
Тут сразу видно, как BM25 пытается найти точное попадание в текст по заданным словам, но так как запросы абстрактные, что не предполагает прямого цитирования, то ответы BM25 выглядят не в попад. А в Dense ответы больше попадают в смысл запроса.

---

**Вывод:** векторный поиск лучше справляется с перефразировками и абстрактными
запросами, BM25  с точными фразами из текста. В идеале стоит комбинировать оба
подхода: BM25 для первичного отбора, Dense для переранжирования. Или выбирать способ, подходящий к конкретной задаче.

## 9. Выводы

Оба метода векторный поиск и BM25 в целом справились хорошо,
особенно на прямых цитатах и сюжетных запросах. Булгаков оказался
отличным источником: текст большой, и почти на любой запрос находится
что-то уместное.

По метрикам BM25 выиграл (MRR 0.932 vs 0.886). Но это объясняется самой метрикой, так как она отслеживает насколько высоко в списке стоит правильный ответ, а BM25 находит их почти дословно. Если посмотреть реальные запросы и ответах, то по смыслу видно, что Dense лучше
справляется с перефразировками и абстрактными вопросами, там где нет ключевых слов напряму в оригинальном тексте.

Главная проблема обоих подходов - если подходящего отрывка в базе нет,
всё равно что-тоо возвращается. Нет "не знаю".
Ещё параграфы иногда обрываются на полуслове (BM25 - 'Ну, будет другой редактор и даже, может быть, еще').

Из улучшений: добавить порог сходства, разбить текст на отдельные
предложения и объединить оба метода.